In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Use the Python executable from the active environment
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# Prefer local loopback for Spark networking
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

builder = SparkSession.builder \
    .appName("PySparkDemo") \
    .master("local[*]") \
    .config("spark.pyspark.python", python_path) \
    .config("spark.pyspark.driver.python", python_path) \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("SparkSession created successfully.")

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [ ]:
data = [
    (101, "Amit", "Laptop", "Electronics", 1, 55000),
    (102, "Neha", "Mobile", "Electronics", 2, 25000),
    (103, "Rahul", "Chair", "Furniture", 4, 3000),
    (104, "Priya", "Table", "Furniture", 2, 8000),
    (105, "Karan", "Headphones", "Electronics", 3, 2000)
]

columns = [
    "OrderID",
    "CustomerName",
    "Product",
    "Category",
    "Quantity",
    "Price"
]

sales_df = spark.createDataFrame(data, columns)

sales_df.show()

In [ ]:
delta_path = "output/delta/retail_sales"

sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [ ]:
delta_df = spark.read \
    .format("delta") \
    .load(delta_path)

delta_df.show()

In [ ]:
parquet_path = "output/parquet/retail_sales"

sales_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(parquet_path)

In [ ]:
delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.update(
    condition=col("OrderID") == 101,
    set={"Price": lit(52000)}
)

In [ ]:
# spark.sql(f"""
#     ALTER TABLE delta.`{delta_path}`
#     ADD CONSTRAINT valid_quantity
#     CHECK (Quantity > 0)
# """)

In [ ]:
# delta_table.update(
#     condition=col("OrderID") == 101,
#     set={"Quantity": lit(-5)}
# )

In [ ]:
delta_table.history() \
    .select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    ) \
    .show(truncate=False)